In [ ]:
import os
import cv2
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Concatenate, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def iou_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


In [ ]:
# Function to split images into tiles
def split_image_into_tiles(image_path, mask_path, tile_size, size):
    img = tifffile.imread(image_path)
    mask = tifffile.imread(mask_path)
    mask = mask[:, :, 0] if len(mask.shape) == 3 else mask

    tiles_img, tiles_mask = [], []
    for x in range(0, img.shape[1], tile_size):
        for y in range(0, img.shape[0], tile_size):
            tile_img = img[y:y+tile_size, x:x+tile_size, :]
            tile_mask = mask[y:y+tile_size, x:x+tile_size]

            tile_img = cv2.resize(tile_img, (size, size))
            tile_mask = cv2.resize(tile_mask, (size, size))
            tile_mask = (tile_mask > 0).astype(np.uint8)

            tiles_img.append(tile_img)
            tiles_mask.append(tile_mask)

    return np.array(tiles_img), np.array(tiles_mask)

# Load dataset
def load_data(image_dir, mask_dir, tile_size=256, size=256):
    images, masks = [], []
    image_filenames = sorted(os.listdir(image_dir))
    mask_filenames = sorted(os.listdir(mask_dir))

    for image_filename in image_filenames:
        if image_filename.endswith(".TIF"):
            mask_filename = image_filename.replace(".TIF", "_mask.TIF")
            if mask_filename in mask_filenames:
                img_path = os.path.join(image_dir, image_filename)
                mask_path = os.path.join(mask_dir, mask_filename)
                img, mask = split_image_into_tiles(img_path, mask_path, tile_size, size)
                images.extend(img)
                masks.extend(mask)
    return np.array(images), np.array(masks)

# Paths
image_dir = "../../datasets/images"
mask_dir = "../../datasets/masks"
size = 256

# Load data
tiles_img, tiles_mask = load_data(image_dir, mask_dir, tile_size=size, size=size)

# Split sets
X_train, X_test, y_train, y_test = train_test_split(tiles_img, tiles_mask, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

# Reshape masks
y_train = y_train[..., np.newaxis]
y_val = y_val[..., np.newaxis]
y_test = y_test[..., np.newaxis]


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, Concatenate
from tensorflow.keras.optimizers import Adam

# ---- VGG16 segmentation model (no U-Net skip connections) ---- #
def vgg16_segmentation(input_size=(256, 256, 3), freeze_encoder=True):
    base = VGG16(weights="imagenet", include_top=False, input_shape=input_size)

    if freeze_encoder:
        for layer in base.layers:
            layer.trainable = False

    x = base.output  # bottleneck

    # Simple decoder: upsample progressively without skip connections
    x = Conv2DTranspose(512, (3, 3), strides=2, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2DTranspose(256, (3, 3), strides=2, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2DTranspose(128, (3, 3), strides=2, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2DTranspose(64, (3, 3), strides=2, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    # Final projection to single-channel mask
    output = Conv2D(1, (1, 1), activation="sigmoid", padding="same")(x)
    output = tf.image.resize(output, (input_size[0], input_size[1]), method="bilinear")

    model = Model(inputs=base.input, outputs=output)
    
    model.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', dice_coef, iou_coef])
    return model

# Usage
size = 256
model = vgg16_segmentation(input_size=(size, size, 3), freeze_encoder=True)
model.summary()


In [ ]:
# LR schedule
def lr_schedule(epoch):
    initial_lr = 1e-4
    decay = 0.9
    return initial_lr * (decay ** (epoch // 10))

lr_scheduler = LearningRateScheduler(lr_schedule)

# Data augmentation
datagen = ImageDataGenerator(rescale=1./255,
                             shear_range=0.2,
                             zoom_range=0.2,
                             horizontal_flip=True,
                             rotation_range=20,
                             width_shift_range=0.2,
                             height_shift_range=0.2,
                             brightness_range=[0.8, 1.2])

# Callbacks
checkpointer = ModelCheckpoint("best_vgg16.h5", monitor="val_dice_coef", mode="max",
                               save_best_only=True, verbose=1)
earlyStopping = EarlyStopping(monitor="val_dice_coef", patience=5, mode="max", verbose=1)


In [ ]:
history = model.fit(datagen.flow(X_train, y_train, batch_size=32),
                    validation_data=(X_val/255.0, y_val),
                    epochs=50,
                    callbacks=[lr_scheduler, earlyStopping, checkpointer])


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample



def bootstrap_confidence_interval(y_true, y_pred, metric_fn, n_bootstraps=100, alpha=0.95):
    """Bootstrap CI + std for a given metric"""
    stats = []
    n = len(y_true)
    for _ in range(n_bootstraps):
        indices = np.random.randint(0, n, n)
        if metric_fn.__name__ == "roc_auc_score":  # ROC-AUC requires probs
            stat = metric_fn(y_true[indices], y_pred[indices])
        else:  # Binary metrics
            stat = metric_fn(y_true[indices], y_pred[indices])
        stats.append(stat)
    
    stats = np.array(stats)
    mean_val = np.mean(stats)
    std_val  = np.std(stats)
    lower = np.percentile(stats, ((1 - alpha) / 2) * 100)
    upper = np.percentile(stats, (alpha + (1 - alpha) / 2) * 100)
    
    return mean_val, std_val, (lower, upper)


# --- Evaluate model ---
loss, acc, dice, iou = model.evaluate(X_test/255.0, y_test)
print(f"Test Loss: {loss:.4f}, Accuracy: {acc:.4f}, Dice: {dice:.4f}, IoU: {iou:.4f}")

# --- Predictions ---
y_pred = model.predict(X_test/255.0)
y_pred_bin = (y_pred > 0.5).astype(np.uint8)

# Flatten
y_true_flat = y_test.flatten()
y_pred_flat = y_pred_bin.flatten()
y_pred_probs = y_pred.flatten()

# --- Metrics ---
metrics = {
    "Accuracy": lambda yt, yp: np.mean(yt == yp),
    "Precision": lambda yt, yp: precision_score(yt, yp),
    "Recall": lambda yt, yp: recall_score(yt, yp),
    "F1-score": lambda yt, yp: f1_score(yt, yp),
    "ROC-AUC": lambda yt, yp: roc_auc_score(yt, yp),
    "Dice": lambda yt, yp: (2*np.sum(yt*yp))/(np.sum(yt)+np.sum(yp)+1e-7),
    "IoU": lambda yt, yp: np.sum(yt*yp)/(np.sum(yt)+np.sum(yp)-np.sum(yt*yp)+1e-7)
}

print("\n📊 Metrics with 95% Confidence Intervals:")
for name, fn in metrics.items():
    if name == "ROC-AUC":
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_probs, fn)
    else:
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_flat, fn)
    print(f"{name}: {mean_val:.4f} ± {std_val:.4f}  (95% CI: {low:.4f} – {high:.4f})")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Flatten ground truth and predictions
y_true_flat = y_test.flatten()
y_pred_flat = (y_pred.flatten() > 0.5).astype(int)  # threshold at 0.5

# Confusion Matrix
cm = confusion_matrix(y_true_flat, y_pred_flat)
print("Confusion Matrix:\n", cm)

# Classification Report
print(classification_report(y_true_flat, y_pred_flat))

# Heatmap
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Forest","Forest"],
            yticklabels=["Non-Forest","Forest"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Plot training history for loss, accuracy, dice, iou
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Metrics')
plt.title('vgg16_U-Net Training History')
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Epochs: 1 to 31 (early stopping at 31)
epochs = range(1, 32)

# Training metrics
train_loss = [
    0.4736, 0.3641, 0.3509, 0.3423, 0.3402, 0.3341, 0.3309, 0.3302, 0.3270, 0.3286,
    0.3300, 0.3242, 0.3211, 0.3152, 0.3219, 0.3202, 0.3131, 0.3168, 0.3178, 0.3166,
    0.3134, 0.3211, 0.3183, 0.3134, 0.3126, 0.3091, 0.3081, 0.3132, 0.3065, 0.3085, 0.3107
]

train_accuracy = [
    0.7565, 0.8242, 0.8347, 0.8381, 0.8402, 0.8422, 0.8442, 0.8447, 0.8462, 0.8451,
    0.8456, 0.8489, 0.8483, 0.8531, 0.8478, 0.8504, 0.8559, 0.8531, 0.8522, 0.8539,
    0.8542, 0.8503, 0.8519, 0.8546, 0.8553, 0.8569, 0.8577, 0.8546, 0.8582, 0.8582, 0.8570
]

train_dice = [
    0.6996, 0.7847, 0.7935, 0.7960, 0.7985, 0.8024, 0.8029, 0.8028, 0.8050, 0.8032,
    0.8053, 0.8085, 0.8099, 0.8132, 0.8084, 0.8093, 0.8172, 0.8106, 0.8134, 0.8119,
    0.8163, 0.8066, 0.8153, 0.8136, 0.8171, 0.8162, 0.8158, 0.8131, 0.8160, 0.8190, 0.8182
]

train_iou = [
    0.5424, 0.6480, 0.6594, 0.6629, 0.6665, 0.6722, 0.6727, 0.6725, 0.6753, 0.6731,
    0.6761, 0.6804, 0.6819, 0.6870, 0.6808, 0.6825, 0.6926, 0.6834, 0.6877, 0.6861,
    0.6912, 0.6784, 0.6902, 0.6874, 0.6929, 0.6911, 0.6914, 0.6872, 0.6917, 0.6957, 0.6939
]

# Validation metrics
val_loss = [
    0.6418, 0.5424, 0.4073, 0.3295, 0.2948, 0.2807, 0.2821, 0.2692, 0.2755, 0.2725,
    0.2671, 0.2697, 0.2678, 0.2674, 0.2661, 0.2655, 0.2579, 0.2649, 0.2655, 0.2588,
    0.2573, 0.2662, 0.2615, 0.2587, 0.2620, 0.2566, 0.2594, 0.2667, 0.2612, 0.2587, 0.2616
]

val_accuracy = [
    0.5956, 0.6794, 0.8260, 0.8684, 0.8721, 0.8834, 0.8823, 0.8851, 0.8780, 0.8809,
    0.8819, 0.8807, 0.8853, 0.8816, 0.8870, 0.8845, 0.8901, 0.8865, 0.8832, 0.8871,
    0.8901, 0.8806, 0.8886, 0.8896, 0.8871, 0.8903, 0.8890, 0.8843, 0.8885, 0.8906, 0.8909
]

val_dice = [
    0.5673, 0.6552, 0.7344, 0.7798, 0.8079, 0.8132, 0.8111, 0.8246, 0.8174, 0.8244,
    0.8286, 0.8283, 0.8236, 0.8253, 0.8272, 0.8317, 0.8353, 0.8267, 0.8255, 0.8354,
    0.8365, 0.8320, 0.8296, 0.8333, 0.8310, 0.8369, 0.8333, 0.8315, 0.8298, 0.8319, 0.8284
]

val_iou = [
    0.3972, 0.4894, 0.5831, 0.6421, 0.6807, 0.6878, 0.6850, 0.7041, 0.6938, 0.7040,
    0.7095, 0.7094, 0.7026, 0.7049, 0.7077, 0.7145, 0.7195, 0.7070, 0.7052, 0.7196,
    0.7212, 0.7144, 0.7112, 0.7166, 0.7134, 0.7219, 0.7167, 0.7143, 0.7115, 0.7144, 0.7095
]

In [ ]:
plt.figure(figsize=(16, 12))

# Plot 1: Loss
plt.subplot(2, 2, 1)
plt.plot(epochs, train_loss, 'bo-', label='Training Loss', linewidth=2)
plt.plot(epochs, val_loss, 'r-o', label='Validation Loss', linewidth=2)
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Accuracy
plt.subplot(2, 2, 2)
plt.plot(epochs, train_accuracy, 'bo-', label='Training Accuracy', linewidth=2)
plt.plot(epochs, val_accuracy, 'r-o', label='Validation Accuracy', linewidth=2)
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 3: Dice Coefficient
plt.subplot(2, 2, 3)
plt.plot(epochs, train_dice, 'bo-', label='Training Dice Coefficient', linewidth=2)
plt.plot(epochs, val_dice, 'r-o', label='Validation Dice Coefficient', linewidth=2)
plt.title('Training and Validation Dice Coefficient', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Dice Coefficient')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 4: IoU
plt.subplot(2, 2, 4)
plt.plot(epochs, train_iou, 'bo-', label='Training IoU', linewidth=2)
plt.plot(epochs, val_iou, 'r-o', label='Validation IoU', linewidth=2)
plt.title('Training and Validation IoU', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('IoU')
plt.legend()
plt.grid(True, alpha=0.3)

# Adjust layout and show
plt.tight_layout()
plt.show()